# Ensemble Methods


In [1]:
import pandas as pd

import numpy as np

## Choose your own Regression

**Working together with your partner**, choose an interesting data set to fit *regression* models for, that is, a data set with a *quantitative* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *regression* models to predict the *quantitative* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a linear regression model
- One partner should fit a k-nearest neighbors regression model

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create two more models**:

- An ensemble model that uses **voting**
- An ensemble model that uses **stacking**

For each of the four models (linear regression, knn regression, voting ensemble, stacking ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test RMSE using cross-validation, and compare the results

Which model seems to work best?


In [2]:
# Use the heart disease data from Practice Activity 12-1
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor

ha = pd.read_csv("../Week 12/ha_1.csv")

# Predict maximum heart rate from the other quantitative variables
X_reg = ha[["age", "sex", "cp", "trtbps", "chol", "restecg"]]
y_reg = ha["thalach"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

linear_model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

knn_regression_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=10)
)

voting_regression_model = VotingRegressor([
    ("linear", linear_model),
    ("knn", knn_regression_model)
])

stacking_regression_model = StackingRegressor(
    estimators=[
        ("linear", linear_model),
        ("knn", knn_regression_model)
    ],
    final_estimator=LinearRegression()
)

regression_models = {
    "Linear regression": linear_model,
    "KNN regression": knn_regression_model,
    "Voting regression": voting_regression_model,
    "Stacking regression": stacking_regression_model
}

# Fit each model and compare predictions for five test observations
regression_predictions = pd.DataFrame({
    "Actual thalach": y_reg_test.iloc[:5].values
})

for name, model in regression_models.items():
    model.fit(X_reg_train, y_reg_train)
    regression_predictions[name] = model.predict(X_reg_test.iloc[:5])

print(regression_predictions.round(1))

# Estimate test RMSE with 10-fold cross-validation
regression_rmse = {}

for name, model in regression_models.items():
    regression_rmse[name] = -cross_val_score(
        model, X_reg_train, y_reg_train, cv=10,
        scoring="neg_root_mean_squared_error"
    ).mean()

regression_results = pd.Series(
    regression_rmse, name="Cross-validated RMSE"
).sort_values()

print("\n", regression_results.round(2))
print("\nBest model:", regression_results.index[0])

   Actual thalach  Linear regression  KNN regression  Voting regression  \
0             114              147.8           138.9              143.4   
1             158              152.5           148.2              150.3   
2             163              155.6           146.1              150.8   
3             128              145.6           129.9              137.7   
4             144              140.4           157.3              148.9   

   Stacking regression  
0                143.9  
1                150.4  
2                151.5  
3                138.9  
4                146.9  

 Voting regression      21.39
Stacking regression    21.58
KNN regression         21.74
Linear regression      21.75
Name: Cross-validated RMSE, dtype: float64

Best model: Voting regression


## Voting in Classification

Suppose you have three models for predicting a categorical target:

- k-nearest neighbors
- logistic regression
- decision tree

How might you apply voting to create an ensemble model? Discuss conceptually with your partner. Create a concrete example to illustrate your method. Hint: There are two basic approaches to voting in classification; can you think of both?

Note: You can assume a binary target, but the general idea of voting applies to multiclass problems. However, if any of the candidate *models* are not naturally multiclass (like logistic regression) then you have to consider things like OvR and OvO as we have discussed.

## Choose your own Classification

**Working together with your partner**, choose an interesting data set to fit *classification* models for, that is, a data set with a *categorical* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *classification* models to predict the *categorical* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a k-nearest neighbors classification model
- One partner should fit another classification model like decision tree or logistic regression

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create an ensemble model that uses *voting***.

For each of the three models (knn classification, other classification, voting ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test macro precision, recall, and F1 score using cross-validation, and compare the results

Which model seems to work best?

**Hint:** [`VotingClassifier`](https://scikit-learn.org/1.5/modules/generated/sklearn.ensemble.VotingClassifier.html) works almost identically to `VotingRegressor`.


In [3]:
# Use the heart disease data from Practice Activity 12-1
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier

# Predict heart disease diagnosis from the quantitative variables
X_class = ha[[
    "age", "sex", "cp", "trtbps", "chol", "restecg", "thalach"
]]
y_class = ha["diagnosis"]

X_class_train, X_class_test, y_class_train, y_class_test = train_test_split(
    X_class, y_class, test_size=0.20, random_state=42, stratify=y_class
)

knn_classification_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=10)
)

logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

# Hard voting uses the majority class. Soft voting averages probabilities.
voting_classification_model = VotingClassifier(
    estimators=[
        ("knn", knn_classification_model),
        ("logistic", logistic_model)
    ],
    voting="soft"
)

classification_models = {
    "KNN classification": knn_classification_model,
    "Logistic regression": logistic_model,
    "Voting classification": voting_classification_model
}

# Fit each model and compare predictions for five test observations
classification_predictions = pd.DataFrame({
    "Actual diagnosis": y_class_test.iloc[:5].values
})

for name, model in classification_models.items():
    model.fit(X_class_train, y_class_train)
    classification_predictions[name] = model.predict(X_class_test.iloc[:5])

print(classification_predictions)

# Estimate macro precision, recall, and F1 with 10-fold cross-validation
classification_results = {}

for name, model in classification_models.items():
    precision = cross_val_score(
        model, X_class_train, y_class_train, cv=10,
        scoring="precision_macro"
    ).mean()
    recall = cross_val_score(
        model, X_class_train, y_class_train, cv=10,
        scoring="recall_macro"
    ).mean()
    f1 = cross_val_score(
        model, X_class_train, y_class_train, cv=10,
        scoring="f1_macro"
    ).mean()
    classification_results[name] = [precision, recall, f1]

classification_results = pd.DataFrame(
    classification_results,
    index=["Precision", "Recall", "F1 score"]
).T

print("\n", classification_results.round(3))
print(
    "\nBest model:",
    classification_results["F1 score"].idxmax()
)

  Actual diagnosis KNN classification Logistic regression  \
0          Disease            Disease          No Disease   
1       No Disease            Disease             Disease   
2          Disease         No Disease          No Disease   
3       No Disease         No Disease          No Disease   
4          Disease            Disease             Disease   

  Voting classification  
0               Disease  
1               Disease  
2            No Disease  
3            No Disease  
4               Disease  

                        Precision  Recall  F1 score
KNN classification         0.764   0.732     0.729
Logistic regression        0.784   0.770     0.772
Voting classification      0.786   0.769     0.768

Best model: Logistic regression
